# Player Level Data

In [ ]:
import pandas as pd
import re
from rapidfuzz import process, fuzz
from unidecode import unidecode

# ── Load ───────────────────────────────────────────────────────────────────
roster = pd.read_csv("player_rosters.csv")
profiles = pd.read_csv(
    "/Users/banna/Desktop/World Cup/football-datasets-main/player_data/player_profiles/player_profiles.csv"
)
performances = pd.read_csv(
    "/Users/banna/Desktop/World Cup/football-datasets-main/player_data/player_performances/player_performances.csv"
)

# ── Filter men's WC only ───────────────────────────────────────────────────
WC_PRIOR_SEASON = {
    "WC-2006": "05/06",
    "WC-2010": "09/10",
    "WC-2014": "13/14",
    "WC-2018": "17/18",
    "WC-2022": "21/22",
}
roster = roster[roster["tournament_id"].isin(WC_PRIOR_SEASON)].copy()
roster["target_season"] = roster["tournament_id"].map(WC_PRIOR_SEASON)

# ── Nickname -> real name lookup ───────────────────────────────────────────
NICKNAME_MAP = {
    "juninho pernambucano": "amarildo tavares da silveira",
    "kaka": "ricardo izecson dos santos leite",
    "ronaldo": "ronaldo luis nazario de lima",
    "ronaldinho": "ronaldo de assis moreira",
    "rivaldo": "rivaldo vitor borba ferreira",
    "adriano": "adriano leite ribeiro",
    "fred": "frederico chaves guedes",
    "hulk": "givanildo vieira de sousa",
    "neymar": "neymar da silva santos junior",
    "robinho": "robinson de souza",
    "diego": "diego ribas da cunha",
    "cafu": "marcos evangelista de morais",
    "roberto carlos": "roberto carlos da silva",
    "bebeto": "jose roberto gama de oliveira",
    "romario": "romario de souza faria",
    "emerson": "emerson ferreira da rosa",
    "gilberto silva": "gilberto aparecido da silva",
    "lucio": "lucimar ferreira da silva",
    "maicon": "maicon douglas sisenando",
    "jo": "joao alves de assis silva",
    "elano": "elano blumer",
    "grafite": "vander luis de souza",
    "pauleta": "pedro miguel pauleta",
    "maniche": "nuno ribeiro maniche",
    "deco": "anderson luis de abreu oliveira",
    "chicharito": "javier hernandez balcazar",
    "kun aguero": "sergio leonel aguero del castillo",
    "gio": "giovanni dos santos",
    "cuauhtemoc blanco": "cuauhtemoc blanco bravo",
    "el hadji diouf": "el hadji ousseynou diouf",
    "didier drogba": "didier yves drogba tebily",
    "yaya toure": "gnegneri yaya toure",
    "kolo toure": "souleymane kolo toure",
    "gervinho": "gervais lombe yao kouassi",
    "samuel etoo": "samuel etoo fils",
    "eto o": "samuel etoo fils",
    "geremi": "geremi sorele njitap fotso",
    "dado prso": "dado prso",
    "rogério ceni": "rogerio ceni",
    "rogério ceni": "rogerio ceni",
}


# ── Normalise names — aggressive: strip accents, hyphens, apostrophes ──────
def norm(s):
    if not pd.notna(s) or str(s).strip() == "":
        return ""
    s = unidecode(str(s)).lower().strip()
    s = re.sub(r"[-'`']", " ", s)  # hyphens and apostrophes -> space
    s = re.sub(r"[^\w\s]", "", s)  # remove remaining punctuation
    s = re.sub(r"\s+", " ", s).strip()
    return s


roster["full_name"] = (
    roster["given_name"].replace("not applicable", "").fillna("")
    + " "
    + roster["family_name"].fillna("")
).str.strip()
roster["full_name_norm"] = roster["full_name"].apply(norm)
roster["dob_clean"] = pd.to_datetime(roster["birth_date"], errors="coerce")

profiles["player_name_clean"] = profiles["player_name"].str.replace(
    r"\s*\(\d+\)\s*$", "", regex=True
)
profiles["full_name_norm"] = profiles["player_name_clean"].apply(norm)
profiles["home_name_norm"] = profiles["name_in_home_country"].apply(norm)
profiles["dob_clean"] = pd.to_datetime(profiles["date_of_birth"], errors="coerce")

tm_index = profiles[
    ["player_id", "player_name_clean", "full_name_norm", "home_name_norm", "dob_clean"]
].copy()
tm_index = tm_index.dropna(subset=["full_name_norm"])
tm_index = tm_index[tm_index["full_name_norm"] != ""].reset_index(drop=True)

tm_names = tm_index["full_name_norm"].tolist()
tm_ids = tm_index["player_id"].tolist()
tm_home_names = tm_index["home_name_norm"].tolist()

# ── Regions unlikely to be in TM — skip aggressive matching ───────────────
LOW_COVERAGE_NATIONS = {
    "Saudi Arabia",
    "Iran",
    "North Korea",
    "South Korea",
    "Japan",
    "South Africa",
    "Togo",
    "Angola",
    "Morocco",
    "Algeria",
    "Egypt",
    "Tunisia",
    "Cameroon",
    "Nigeria",
    "Ghana",
    "Ivory Coast",
    "Senegal",
    "Trinidad and Tobago",
    "Honduras",
    "Costa Rica",
    "Ecuador",
    "Panama",
    "Qatar",
    "New Zealand",
}


# ── Match roster -> TM profiles ────────────────────────────────────────────
def try_match_df(name, candidates_df, scorer, cutoff):
    if candidates_df.empty or not name:
        return None
    result = process.extractOne(
        name,
        candidates_df["full_name_norm"].tolist(),
        scorer=scorer,
        score_cutoff=cutoff,
    )
    if result:
        _, score, idx = result
        return int(candidates_df.iloc[idx]["player_id"]), score
    return None


matched_rows = []

for _, row in roster.iterrows():
    name = row["full_name_norm"]
    dob = row["dob_clean"]
    nation = row["team_name"]
    resolved = False

    # Apply nickname map
    search_name = NICKNAME_MAP.get(name, name)

    dob_candidates = pd.DataFrame()
    if pd.notna(dob):
        dob_candidates = tm_index[tm_index["dob_clean"] == dob].reset_index(drop=True)

    # S1: DOB + name (threshold 85)
    if not dob_candidates.empty:
        res = try_match_df(search_name, dob_candidates, fuzz.token_sort_ratio, 85)
        if res:
            matched_rows.append(
                {
                    **row.to_dict(),
                    "tm_player_id": res[0],
                    "match_score": res[1],
                    "match_method": "dob+name",
                }
            )
            resolved = True

    # S2: DOB + relaxed threshold (70 — safe because DOB anchors)
    if not resolved and not dob_candidates.empty:
        res = try_match_df(search_name, dob_candidates, fuzz.token_sort_ratio, 70)
        if res:
            matched_rows.append(
                {
                    **row.to_dict(),
                    "tm_player_id": res[0],
                    "match_score": res[1],
                    "match_method": "dob+relaxed",
                }
            )
            resolved = True

    # S3: DOB + last name only
    if not resolved and not dob_candidates.empty:
        last = norm(row["family_name"])
        if last:
            dob_candidates["last_norm"] = dob_candidates["full_name_norm"].apply(
                lambda x: x.split()[-1] if x.strip() else ""
            )
            result = process.extractOne(
                last,
                dob_candidates["last_norm"].tolist(),
                scorer=fuzz.ratio,
                score_cutoff=92,
            )
            if result:
                _, score, idx = result
                matched_rows.append(
                    {
                        **row.to_dict(),
                        "tm_player_id": int(dob_candidates.iloc[idx]["player_id"]),
                        "match_score": score,
                        "match_method": "dob+lastname",
                    }
                )
                resolved = True

    # S4: DOB + home country name
    if not resolved and not dob_candidates.empty:
        home_candidates = dob_candidates.copy()
        home_candidates["full_name_norm"] = home_candidates["home_name_norm"].apply(
            norm
        )
        home_candidates = home_candidates[
            home_candidates["full_name_norm"] != ""
        ].reset_index(drop=True)
        res = try_match_df(search_name, home_candidates, fuzz.token_sort_ratio, 85)
        if res:
            matched_rows.append(
                {
                    **row.to_dict(),
                    "tm_player_id": res[0],
                    "match_score": res[1],
                    "match_method": "dob+homename",
                }
            )
            resolved = True

        # S5: name-only fuzzy — high threshold prevents false positives
        if not resolved:
            result = process.extractOne(
                search_name, tm_names, scorer=fuzz.token_sort_ratio, score_cutoff=92
            )
            if result:
                _, score, idx = result
                matched_rows.append(
                    {
                        **row.to_dict(),
                        "tm_player_id": int(tm_ids[idx]),
                        "match_score": score,
                        "match_method": "name_only",
                    }
                )
                resolved = True

        # S6: home name only
        if not resolved:
            result = process.extractOne(
                search_name,
                tm_home_names,
                scorer=fuzz.token_sort_ratio,
                score_cutoff=92,
            )
            if result:
                _, score, idx = result
                matched_rows.append(
                    {
                        **row.to_dict(),
                        "tm_player_id": int(tm_ids[idx]),
                        "match_score": score,
                        "match_method": "home_only",
                    }
                )
                resolved = True

matched = pd.DataFrame(matched_rows)
print(matched["match_method"].value_counts())
print(f"\nMatch rate: {matched['tm_player_id'].notna().mean():.1%}")
print(f"\nUnmatched by nation (top 15):")
print(
    matched[matched["match_method"] == "unmatched"]["team_name"].value_counts().head(15)
)

# ── Pull prior-season performances ─────────────────────────────────────────
performances["player_id"] = performances["player_id"].astype(int)

lookup = matched[matched["tm_player_id"].notna()][
    ["tournament_id", "target_season", "tm_player_id", "position_code"]
].copy()
lookup["tm_player_id"] = lookup["tm_player_id"].astype(int)

perf = performances.merge(
    lookup,
    left_on=["player_id", "season_name"],
    right_on=["tm_player_id", "target_season"],
    how="inner",
)


# ── Aggregate: one row per player per tournament ───────────────────────────
def agg_performances(df):
    primary = df.sort_values("minutes_played", ascending=False).iloc[0]
    return pd.Series(
        {
            "competition_name": primary["competition_name"],
            "club_name": primary["team_name"],
            "minutes_played": df["minutes_played"].sum(),
            "appearances": df["nb_on_pitch"].sum(),
            "goals": df["goals"].sum(),
            "assists": df["assists"].sum(),
            "own_goals": df["own_goals"].sum(),
            "penalty_goals": df["penalty_goals"].sum(),
            "yellow_cards": df["yellow_cards"].sum(),
            "second_yellow_cards": df["second_yellow_cards"].sum(),
            "direct_red_cards": df["direct_red_cards"].sum(),
            "goals_conceded": df["goals_conceded"].sum(),
            "clean_sheets": df["clean_sheets"].sum(),
        }
    )


perf_agg = (
    perf.groupby(["tm_player_id", "tournament_id"])
    .apply(agg_performances, include_groups=False)
    .reset_index()
)

# ── Final join ─────────────────────────────────────────────────────────────
final = matched.merge(perf_agg, on=["tm_player_id", "tournament_id"], how="left")


# ── Flag missingness reasons ───────────────────────────────────────────────
def missingness_reason(row):
    if row["match_method"] == "unmatched" and row["team_name"] in LOW_COVERAGE_NATIONS:
        return "low_coverage_nation"
    if row["match_method"] == "unmatched":
        return "not_in_tm"
    if pd.isna(row["minutes_played"]):
        return "matched_no_perf"
    return "ok"


final["data_status"] = final.apply(missingness_reason, axis=1)
final["perf_found"] = final["data_status"] == "ok"

print(f"\nData status breakdown:")
print(final["data_status"].value_counts())
print(
    f"\nPerformance coverage: {final['perf_found'].sum()}/{len(final)} ({final['perf_found'].mean():.1%})"
)

final.head(10)

/var/folders/2_/skwfrb393zjdzz43_1hyfg8m0000gq/T/ipykernel_8951/3364940038.py:8: DtypeWarning: Columns (29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  profiles     = pd.read_csv('/Users/banna/Desktop/World Cup/football-datasets-main/player_data/player_profiles/player_profiles.csv')


match_method
dob+name        2702
name_only         95
dob+lastname      42
dob+relaxed       42
dob+homename       7
home_only          2
Name: count, dtype: int64

Match rate: 100.0%

Unmatched by nation (top 15):
Series([], Name: count, dtype: int64)

Data status breakdown:
data_status
ok                 2664
matched_no_perf     226
Name: count, dtype: int64

Performance coverage: 2664/2890 (92.2%)


,key_id,tournament_id,tournament_name,team_id,team_name,team_code,player_id,family_name,given_name,shirt_number,...,assists,own_goals_y,penalty_goals_y,yellow_cards_y,second_yellow_cards,direct_red_cards,goals_conceded,clean_sheets,data_status,perf_found
0,8294,WC-2006,2006 FIFA Men's World Cup,T-02,Angola,AGO,P-24930,Ricardo,João,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,matched_no_perf,False
1,8298,WC-2006,2006 FIFA Men's World Cup,T-02,Angola,AGO,P-44205,Kali,not applicable,5,...,0.0,0.0,0.0,4.0,1.0,0.0,0.0,0.0,ok,True
2,8301,WC-2006,2006 FIFA Men's World Cup,T-02,Angola,AGO,P-88171,Macanga,André,8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,matched_no_perf,False
3,8302,WC-2006,2006 FIFA Men's World Cup,T-02,Angola,AGO,P-86226,Mantorras,not applicable,9,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ok,True
4,8304,WC-2006,2006 FIFA Men's World Cup,T-02,Angola,AGO,P-22865,Mateus,not applicable,11,...,2.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,ok,True
5,8308,WC-2006,2006 FIFA Men's World Cup,T-02,Angola,AGO,P-40794,Marques,Rui,15,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ok,True
6,8309,WC-2006,2006 FIFA Men's World Cup,T-02,Angola,AGO,P-78483,Flávio,not applicable,16,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,ok,True
7,8312,WC-2006,2006 FIFA Men's World Cup,T-02,Angola,AGO,P-31166,Buengo,Titi,19,...,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,ok,True
8,8316,WC-2006,2006 FIFA Men's World Cup,T-02,Angola,AGO,P-91786,Abreu,Marco,23,...,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,ok,True
9,8320,WC-2006,2006 FIFA Men's World Cup,T-03,Argentina,ARG,P-78133,Coloccini,Fabricio,4,...,0.0,0.0,0.0,6.0,1.0,2.0,0.0,0.0,ok,True


In [ ]:
final = final.rename(
    columns={
        "goals_x": "wc_goals",
        "goals_y": "club_goals",
        "own_goals_x": "wc_own_goals",
        "own_goals_y": "club_own_goals",
        "penalty_goals_x": "wc_penalty_goals",
        "penalty_goals_y": "club_penalty_goals",
        "yellow_cards_x": "wc_yellow_cards",
        "yellow_cards_y": "club_yellow_cards",
    }
)

In [42]:
final.to_csv("player_stats_prior_season.csv", index=False)